In [ ]:
import zarr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

In [ ]:
ZARR_ROOT = r"C:\Users\ChenJeff\Documents\ves_zarrs2"

SCROLLS = [
    {"id": 20260115000000, "name": "w044", "scroll": "PHerc0139"},
    {"id": 20250223000000, "name": "w059", "scroll": "PHerc0139"},
    {"id": 20260206000001, "name": "w047", "scroll": "PHerc0139"},
    {"id": 20260716083545, "name": "test", "scroll": "PHerc0813"},
]

zarrs = {}
for s in SCROLLS:
    path = rf"{ZARR_ROOT}\{s['id']}.zarr"
    z = zarr.open(path, mode="r")
    zarrs[s["id"]] = z
    print(f"{s['name']:5s} ({s['id']})  shape={z.shape}  dtype={z.dtype}")

In [ ]:
# extract mid-depth slice for each zarr (shape is Z,Y,X so axis 0 is depth)
slices = {}
for s in SCROLLS:
    z = zarrs[s["id"]]
    mid = z.shape[0] // 2
    slices[s["id"]] = z[mid].astype(np.float32)
    print(f"{s['name']:5s}  mid_depth={mid}  slice shape={slices[s['id']].shape}")

In [ ]:
def show_slice(scroll_id, name, scroll, downsample=2):
    """show mid-depth slice at 1/downsample resolution, sized to aspect ratio"""
    img = slices[scroll_id][::downsample, ::downsample]
    lo, hi = np.percentile(img, 1), np.percentile(img, 99)
    img_n = np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1)
    h, w = img_n.shape
    fig, ax = plt.subplots(figsize=(20, 20 * h / w))
    ax.imshow(img_n, cmap="gray", vmin=0, vmax=1)
    ax.set_title(
        f"{name} ({scroll}) — {scroll_id} — half-res mid slice  [{h}x{w} displayed / {h*2}x{w*2} full]",
        fontsize=11
    )
    ax.axis("off")
    plt.tight_layout()
    plt.show()

show_slice(20260115000000, "w044", "PHerc0139")

In [ ]:
show_slice(20250223000000, "w059", "PHerc0139")

In [ ]:
show_slice(20260206000001, "w047", "PHerc0139")

In [ ]:
show_slice(20260716083545, "test", "PHerc0813")

In [ ]:
print(f"{'name':<8} {'scroll':<12} {'shape':<18} {'mean':>8} {'std':>8} {'p1':>8} {'p99':>8} {'max':>8}")
print("-" * 82)
for s in SCROLLS:
    img = slices[s["id"]]
    # mask out exact-zero pixels (outside papyrus footprint)
    valid = img[img > 0]
    p1, p99 = np.percentile(valid, 1), np.percentile(valid, 99)
    print(f"{s['name']:<8} {s['scroll']:<12} {str(img.shape):<18}"
          f" {valid.mean():>8.1f} {valid.std():>8.1f} {p1:>8.1f} {p99:>8.1f} {valid.max():>8.1f}")